In [2]:
import pandas as pd
import numpy as np

## EXERCISE QUIZ: Sprinkler example (modified)
Repeat the third exercise about the Bayesian network (i.e., Sprinkler example)

Compute the probability distribution of the wet grass given that is cloudy with the following data (NB: Some values are changed)

<img src='images/sprinkler2.png'>


In [18]:
t, f = 0, 1

P_R_C = np.array([[0.8,0.1],
                  [0.2,0.9]])
P_S_C = np.array([[0.1,0.5],
                  [0.9,0.5]])
P_W_RS = np.array([[[0.95, 0.90], [0.75, 0.80]],
                   [[0.05, 0.10], [0.25, 0.20]]]) 

# Method 1: Direct computation
P_W = (P_W_RS[:,t,t] * P_R_C[t,t] * P_S_C[t,t] +  # P(W|r,s)P(r|c)P(s|c)
       P_W_RS[:,t,f] * P_R_C[t,t] * P_S_C[f,t] +  # P(W|-r,s)P(-r|c)P(s|c)
       P_W_RS[:,f,t] * P_R_C[f,t] * P_S_C[t,t] +  # P(W|r,-s)P(r|c)P(-s|c)
       P_W_RS[:,f,f] * P_R_C[f,t] * P_S_C[f,t])   # P(W|-r,-s)P(-r|c)P(-s|c)
print("P(W) by direct computation:", P_W)

# Method 2: Marginalize one variable at a time 
P_W_R = P_W_RS[:,:,t] * P_S_C[t,t] + P_W_RS[:,:,f] * P_S_C[f,t] # P(W|R,s) * P(s|c) + P(W|R,-s) * P(-s|c)= P(W|R)
P_W = P_W_R[:,t] * P_R_C[t,t] + P_W_R[:,f] * P_R_C[f,t]
print("P(W) using intermediate variables:", P_W)

P(W) by direct computation: [0.883 0.117]
P(W) using intermediate variables: [0.883 0.117]


## EXERCISE 1: Weather's probability
You are given a (fake) <a href="https://drive.google.com/file/d/1LjZLE9ozaHcBwiCl90mHaS1nXKcglfr4/view">padua_weather.csv</a>
of historical records for Padua's weather. The weather, which can be either rainy (= 1 in the dataset), misty (= 2), or sunny (= 3), is reported for each day of the week, for a whole year (52 weeks).

After you formalised the problem (i.e. identify the random variables and necessary mathematical formulae), write a Python program that reads the dataset and computes the following:
- probability of being sunny during the weekend (one or both days);
- expected weather for each day of the week (*);
- supposed you don't know which day of the week is today: although very unrealistic, how could you guess which day is today based only on the weather?

(\*) An expected value of, for example, 2.5 can be interpreted as "a mix of misty and sunny weather".

In [4]:
df = pd.read_csv('padua_weather.csv')

RAINY = 1
MISTY = 2
SUNNY = 3
df.head()


,Monday,Tuesday,Wednesday,Thursday,Friday,Saturday,Sunday
0,1,2,1,1,1,2,1
1,2,1,2,1,1,2,1
2,2,1,2,2,1,1,1
3,1,1,1,3,3,2,1
4,3,3,3,3,1,3,2


In [5]:
df2 = df.copy()
df2['sunny_weekend'] = (df['Saturday'] == SUNNY) | (df['Sunday'] == SUNNY)
df2['sunny_weekend'].mean()

np.float64(0.4423076923076923)

In [6]:
expected_weather = df.mean()
expected_weather

Monday       2.076923
Tuesday      1.980769
Wednesday    2.038462
Thursday     1.942308
Friday       1.961538
Saturday     1.846154
Sunday       1.750000
dtype: float64

In [7]:
weather_weekly_count = df.apply(pd.value_counts).fillna(0)
weather_weekly_count


for w in [RAINY,MISTY,SUNNY]:
    row = weather_weekly_count.loc[w]
    row = row / row.sum()
    print(f'Weather {w} probabilities: ')
    print(row)
    print(row.idxmax())
    print('__________')

Weather 1 probabilities: 
Monday       0.124088
Tuesday      0.138686
Wednesday    0.116788
Thursday     0.138686
Friday       0.138686
Saturday     0.145985
Sunday       0.197080
Name: 1, dtype: float64
Sunday
__________
Weather 2 probabilities: 
Monday       0.126126
Tuesday      0.135135
Wednesday    0.162162
Thursday     0.153153
Friday       0.144144
Saturday     0.180180
Sunday       0.099099
Name: 2, dtype: float64
Saturday
__________
Weather 3 probabilities: 
Monday       0.181034
Tuesday      0.155172
Wednesday    0.155172
Thursday     0.137931
Friday       0.146552
Saturday     0.103448
Sunday       0.120690
Name: 3, dtype: float64
Monday
__________


C:\Users\tomma\AppData\Local\Temp\ipykernel_23032\1152394441.py:1: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  weather_weekly_count = df.apply(pd.value_counts).fillna(0)


# SOLUTIONS

## Probability of being sunny during the weekend (one or both days)

We have two variables, $W = \{1, 2, 3\}$ and $D = \{mo, tu, we, th, fr, sa, su\}$.

There are 23 out of 52 weekends in which one of the two days is sunny, therefore

$P(W = 3 | D = sa \vee D = su) = 23/52 \approx 0.4423$

<!-- ## Expected weather for each day of the week -->

There are 17 rainy, 14 misty, and 21 sunny Mondays out of 52, therefore

$
\begin{align}
E(W | D = mo) &= \sum_y y~P(W = y | D = mo) \nonumber\\
&= 1 \times P(W = 1 | D = mo) + 2 \times P(W = 2 | D = mo) + 3 \times P(W = 3 | D = mo) \nonumber\\
&= 1 \times (17/52) + 2 \times (14/52) + 3  \times (21/52) \nonumber\\
&\approx 2.077 \nonumber
\end{align}
$

This is equivalent to sum all the values of the Monday column and divide by the number of rows.

Same reasoning for all the other days.


## Guess which day is today based only on the weather

If it's rainy, we need to compute the conditional probability distributions of the days given $W=1$

${\bf P}(D | W=1) = \langle P(mo | W=1), P(tu | W=1), P(we | W=1), P(th | W=1), P(fr | W=1), P(sa | W=1), P(su | W=1)\rangle$

and then choose the day with the highest probability.

In the dataset, out of 137 rainy days, 17 are Mondays, 19 Tuesdays, 16 Wednesdays, 19 Thursdays, 19 Fridays, 20 Saturdays and 27 Sundays. Therefore

$
\begin{align}
{\bf P}(D | W=1) &= \langle (17/137), (19/137), (16/137), (19/137), (19/137), (20/137), (27/137) \rangle \nonumber\\
&\approx \langle 0.124, 0.139, 0.117, 0.139, 0.139, 0.146, {\bf 0.197} \rangle \nonumber
\end{align}
$

Since the last one is the biggest, if it's raining, it's most likely Sunday.

Same reasoning for the other weather conditions.

PS: notice we could get to the same conclusions simply counting and comparing the number of rainy Mondays, Tuesdays, etc. without normalising by the total number 137 (but then of course it wouldn't be a *probability* distribution).

## EXERCISE 2: Broad Street cholera outbreak

The following is a simplified version of an example in Judea Pearl's *The Book of Why*. It refers to a case of cholera epidemic, caused by contaminated water, which killed hundreds of people in London between 1853 and 1854. The diagram below illustrates some of the key factors explaining this epidemic, in particular:
- $X$ indicates whether the water company's intake was downstream of the London's sewers;
- $W$ indicates whether the water was contaminated or not;
- $Z$ indicates the presence of other external factors (e.g. poverty, miasma, etc.);
- $Y$ indicates the outbreak of cholera.

<img src='images/cholera.png'>

(please note the probabilities in the diagram are fake)

> - Formalise the problem using opportune mathematical notations and derive an expression for computing the probability distribution of the cholera given that the water company's intake is upstream (i.e. what is the query? how can it be decomposed?)
> - Write a Python program that computes the actual probabilities of the above distribution using the information from the given CPTs.

In [20]:
import numpy as np
t,f = 0,1

p_x = [0.5,0.5]
p_z = [0.25,0.75]

p_w_xz = np.array([[[0.90,0.85],
           [0.10,0.02]],
          [[0.10,0.15],
           [0.90,0.98]]])

p_y_wz = np.array([[[0.80,0.75],
                    [0.15,0.05]],
                    [[0.20,0.25],
                    [0.85,0.95]]
                  ])


## SOLUTIONS: Probability distribution of the cholera given that the water company's intake is upstream

The question is to find the probability distribution ${\bf P}(Y | \neg x)$, where $Y$ is the cholera variable, and $\neg x$ is the conditional event "not downstream". As usual, we can get this from the joint probability distribution of the Bayesian network, summing out the hidden variables ($Z$ and $W$ in this case) and normalising:

$
\begin{align}
{\bf P}(Y | \neg x) &= \alpha {\bf P}(Y, \neg x) \nonumber\\
&= \alpha \sum_{z, w} {\bf P}(Y, w, \neg x, z) \nonumber\\
&= \alpha \sum_z \sum_w {\bf P}(Y | w, z) P(w | \neg x, z) P(\neg x) P(z) \nonumber\\
&= \alpha P(\neg x) \sum_z P(z) \sum_w {\bf P}(Y | w, z) P(w | \neg x, z) \nonumber
\end{align}
$


## Computes the actual probabilities of the above distribution

The Python program should take the probabilities from the CPTs and compute the following numbers:

$
\begin{align}
{\bf P}(Y | \neg x) &= \alpha \times 0.5 \times \Big[ 0.25 \times \big( \langle 0.8, 0.2 \rangle \times 0.10 + \langle 0.15, 0.85 \rangle \times 0.90 \big) + 0.75 \times \big( \langle 0.75, 0.25 \rangle \times 0.02 + \langle 0.05, 0.95 \rangle \times 0.98 \big) \Big] \nonumber\\
&= \alpha \times 0.5 \times \Big[ 0.25 \times \big( \langle 0.08, 0.02 \rangle + \langle 0.135, 0.765 \rangle \big) + 0.75 \times \big( \langle 0.015, 0.005 \rangle + \langle 0.049, 0.931 \rangle \big) \Big] \nonumber\\
&= \alpha \times 0.5 \times \Big[ 0.25 \times \langle 0.215, 0.785 \rangle + 0.75 \times \langle 0.064, 0.936 \rangle \Big] \nonumber\\
&= \alpha \times 0.5 \times \Big[ \langle 0.05375, 0.19625 \rangle + \langle 0.048, 0.702 \rangle \Big] \nonumber\\
&= \alpha \times 0.5 \times \langle 0.10175, 0.89825 \rangle \nonumber\\
&= \langle 0.10175, 0.89825 \rangle \nonumber
\end{align}
$

$
P(A|X) = P(A|x=true)*p(x=true) + P(A|x=false)*p(x=false)
$

In [21]:

p_y = p_y_wz[:,t,t] * p_w_xz[t,f,t] * p_z[t] + \
      p_y_wz[:,t,f] * p_w_xz[t,f,f] * p_z[f] + \
      p_y_wz[:,f,t] * p_w_xz[f,f,t] * p_z[t] + \
      p_y_wz[:,f,f] * p_w_xz[f,f,f] * p_z[f]

p_y_wz[:,t,t] * p_w_xz[t,f,t] * p_z[t] # P(Y|W=true,Z=true) * P(W=true|X=false,Z=true) * P(Z=true)
p_y_wz[:,t,f] * p_w_xz[t,f,f] * p_z[f] # P(Y|W=true,Z=false) * P(W=true|X=false,Z=false) * P(Z=false)
p_y_wz[:,f,t] * p_w_xz[f,f,t] * p_z[t] # P(Y|W=false,Z=true) * P(W=false|X=false,Z=true) * P(Z=true)
p_y_wz[:,f,f] * p_w_xz[f,f,f] * p_z[f] # P(Y|W=false,Z=false) * P(W=false|X=false,Z=false) * P(Z=false)
p_y

array([0.10175, 0.89825])

In [ ]:

print('P(W|w,-z)=',p_w_xz[t,t,f]) # P(w|x,-z)
print('P(W|-w,z)=',p_y_wz[t,f,t]) # P(-y|-w,z)

print('P(Y|w, x = false, z = true)= ', p_y_wz[:,t,f]*p_w_xz[t,f,f]+ p_y_wz[:,f,f]*p_w_xz[f,f,f])
print('P(Y|w, x = false, z = true)= ', p_y_wz[:,t,t]*p_w_xz[t,f,t]+ p_y_wz[:,f,t]*p_w_xz[f,f,t])

p_y = (p_y_wz[:,t,t] * p_w_xz[t,f,t] * p_z[t] + # P(Y|w,z) * P(w|-x,z) * P(z)
       p_y_wz[:,t,f] * p_w_xz[t,f,f] * p_z[f] + # P(Y|w,-z) * P(w|-x,-z) * P(-z)
       p_y_wz[:,f,t] * p_w_xz[f,f,t] * p_z[t] + # P(Y|-w,z) * P(-w|-x,z) * P(z)
       p_y_wz[:,f,f] * p_w_xz[f,f,f] * p_z[f])  # P(Y|-w,-z) * P(-w|-x,-z) * P(-z)

print(f'p(Y|w,z,x=false) =  {p_y}')


P(W|w,-z)= 0.85
P(W|-w,z)= 0.15
P(Y|w, x = false, z = true)=  [0.064 0.936]
P(Y|w, x = false, z = true)=  [0.215 0.785]
p(Y|w,z,x=false) =  [0.10175 0.89825]
